# Exoplanet transits: modelling with batman

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import lightkurve as lk
import batman
from astropy.timeseries import LombScargle

## 1. Choose a star and download its latest TESS light curve

In [ ]:
star = "WASP-18"   # <-- change the star name here

search = lk.search_lightcurve(star, mission="TESS", author="SPOC", exptime=120)
print(search)

lc = search[-1].download()   # last entry = latest sector
lc = lc.remove_nans().normalize().remove_outliers(sigma_upper=4, sigma_lower=20)

t = np.array(lc.time.value, dtype=float)
f = np.array(lc.flux.value, dtype=float)

plt.figure(figsize=(10, 4))
plt.scatter(t, f, s=1, color="gray")
plt.xlabel("Time (BTJD)"); plt.ylabel("Normalized flux"); plt.title(star)
plt.show()

## 2. Find the period with Lomb-Scargle

In [ ]:
# Remove slow trends, then run Lomb-Scargle
f_flat = np.array(lc.flatten(window_length=901).flux.value, dtype=float)

frequency, power = LombScargle(t, f_flat).autopower(
    minimum_frequency=1/15, maximum_frequency=1/0.3, samples_per_peak=1000)
period = 1 / frequency[np.argmax(power)]

# Lomb-Scargle sometimes finds half the true period.
# If the phased plot in the next block shows two dips, change 1 to 2.
period = period * 1

plt.figure(figsize=(10, 4))
plt.plot(1/frequency, power)
plt.xscale("log")
plt.xlabel("Period (days)"); plt.ylabel("Power")
plt.show()
print("Period =", period, "days")

## 3. Find T0 and phase-fold

In [ ]:
# Rough T0 = time of the lowest point
t0 = t[np.argmin(f_flat)]

# Better T0 = centre of all the in-transit points
phase = ((t - t0) / period + 0.5) % 1 - 0.5
depth = 1 - np.percentile(f_flat, 1)
in_transit = f_flat < 1 - depth/2
t0 = t0 + np.median(phase[in_transit]) * period
print("T0 =", t0, "BTJD")

# Flatten again with the transits masked, so the dip keeps its full depth
duration = 1.5 * (np.percentile(phase[in_transit], 95) - np.percentile(phase[in_transit], 5)) * period
mask = lc.create_transit_mask(period=period, transit_time=t0, duration=duration)
f_flat = np.array(lc.flatten(window_length=901, mask=mask).flux.value, dtype=float)

# Phase-fold: mid-transit at phase 0
phase = ((t - t0) / period + 0.5) % 1 - 0.5

plt.figure(figsize=(10, 4))
plt.scatter(phase, f_flat, s=1, color="gray", alpha=0.5)
plt.xlabel("Phase"); plt.ylabel("Normalized flux")
plt.title(f"{star}  |  P = {period:.5f} d,  T0 = {t0:.4f} BTJD")
plt.show()

## 4. Synthetic transit with batman: change the parameters and re-run

In [ ]:
# ---- Change these and re-run this block and the next one ----
params = batman.TransitParams()
params.t0  = 0.0          # mid-transit (keep 0: the data is already centred)
params.per = period       # orbital period in days (from Lomb-Scargle)
params.rp  = 0.097        # planet radius / star radius
params.a   = 3.5          # orbit size / star radius
params.inc = 84.0         # inclination in degrees (90 = edge-on)
params.ecc = 0.0          # eccentricity
params.w   = 90.0         # argument of periastron in degrees
params.limb_dark = "quadratic"
params.u   = [0.4, 0.2]   # limb darkening

model_phase = np.linspace(-0.5, 0.5, 5000)
model_flux = batman.TransitModel(params, model_phase * period).light_curve(params)

plt.figure(figsize=(10, 4))
plt.plot(model_phase, model_flux, color="red")
plt.xlim(-0.15, 0.15)
plt.xlabel("Phase"); plt.ylabel("Relative flux"); plt.title("Synthetic transit (batman)")
plt.show()

## 5. Overlay the model on the data

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(phase, f_flat, s=1, color="gray", alpha=0.5, label="TESS data")
plt.plot(model_phase, model_flux, color="red", lw=2, label="batman model")
plt.xlim(-0.15, 0.15)   # zoom on the transit
plt.xlabel("Phase"); plt.ylabel("Normalized flux")
plt.title(f"{star}  |  Rp/R* = {params.rp},  a/R* = {params.a},  i = {params.inc}°")
plt.legend()
plt.show()